In [26]:
!CUDA_VISIBLE_DEVICES=1 bash run_train_v6b.sh

  RFT-LM Pilot v6b — M=256 (OCR disambiguation test)
  Resume from: /zeng_gk/Amine/Huawei Challenge/RFT/AmineHL/mixed_pilot_v3_seed42/rft_lm/best_model.pt
  Output:      /zeng_gk/Amine/Huawei Challenge/RFT/AmineHL/mixed_pilot_v6b_m256_seed42
  GPU:         1

  Same as v6 except:
    mem_top_m:  64  → 256 (4x more candidates to disambiguate)
  Hypothesis: OCR will help when M is large enough


  Training: RFT-LM
  GPUs: 1, Device: cuda:0
[DATA] Loading /zeng_gk/Amine/Huawei Challenge/RFT/AmineHL/data/c4_train.jsonl...
[DATA] Loaded 5,000 documents in 0.1s
[DATA] Tokenizing...
[DATA] Using GPT-2 BPE tokenizer from /zeng_gk/Amine/Huawei Challenge/RFT/AmineHL/gpt2_tokenizer (vocab=50257)
[DATA] 2,449,189 tokens, 1,195 sequences of length 2048 in 7.3s
[MODEL] rft_lm: 128,359,685 params
[MODEL] d_model=768, n_layers=12, n_heads=12, window=512
[MODEL] memory_layer_idx=6, mem_top_m=256, ocr_dim=256
[MODEL] use_memory=True
[RESUME] Loaded checkpoint from /zeng_gk/Amine/Huawei Challenge/RFT/Ami

In [27]:
!CUDA_VISIBLE_DEVICES=0 python niah_sanity.py \
    --rft_ckpt "/zeng_gk/Amine/Huawei Challenge/RFT/AmineHL/mixed_pilot_v6b_m256_seed42/rft_lm/best_model.pt" \
    --baseline_ckpt "/zeng_gk/Amine/Huawei Challenge/RFT/AmineHL/mixed_pilot_v3_seed42/rft_lm/best_model.pt" \
    --tokenizer_path "/zeng_gk/Amine/Huawei Challenge/RFT/AmineHL/gpt2_tokenizer" \
    --distractor_path "/zeng_gk/Amine/Huawei Challenge/RFT/AmineHL/data/c4_train.jsonl" \
    --seq_len 1024 --depth 0.5 --n_trials 20 --chunk_size 512


[SANITY] tokenizer eos_id=50256
[SANITY] seq_len=1024 depth=0.5 n_trials=20
[SANITY] distractor tokens: 91,214

=== BASELINE ===
  [0] ok=False contains_val=False ref=45
      tail_ctx=...' Billy, Murfreesboro, and Jerry Thompson, Nashville; daughters, Elaine.\nResults for Restaurant in Westmoreland, TN. Get free custom quotes, customer reviews, prices, contact details, opening hours from Westmoreland, TN based businesses with Restaurant keyword. Check out these two awesome ladies!!\nThey competed all weekend at the Heart of\nThe magic number alpha838100 is'
      pred='the most of the most of the most of the most of the most of the most of the most'
  [1] ok=False contains_val=False ref=67
      tail_ctx=...' own steward to ensure alignment.\nWalmart having a mission to help its customers save money and live better is one thing, but putting that promise into coherent action is another thing entirely. Many retailers take such coordinated action for granted, which is why so many legacy 

In [28]:
!SKIP_GATE=1 CUDA_VISIBLE_DEVICES=1 bash run_eval_v6b.sh

[EVAL_V6B] Checkpoints verified.

  STEP 1: Sanity check (v6b, M=256)
[SANITY] tokenizer eos_id=50256
[SANITY] seq_len=1024 depth=0.5 n_trials=20
[SANITY] distractor tokens: 91,214

=== BASELINE ===
  [0] ok=False contains_val=False ref=45
      tail_ctx=...' Billy, Murfreesboro, and Jerry Thompson, Nashville; daughters, Elaine.\nResults for Restaurant in Westmoreland, TN. Get free custom quotes, customer reviews, prices, contact details, opening hours from Westmoreland, TN based businesses with Restaurant keyword. Check out these two awesome ladies!!\nThey competed all weekend at the Heart of\nThe magic number alpha838100 is'
      pred='the most of the most'
  [1] ok=False contains_val=False ref=67
      tail_ctx=...' own steward to ensure alignment.\nWalmart having a mission to help its customers save money and live better is one thing, but putting that promise into coherent action is another thing entirely. Many retailers take such coordinated action for granted, which is why so ma

In [29]:
!CUDA_VISIBLE_DEVICES=1 VARIANT=noalign nohup bash run_train_mt_lm.sh \
    > /tmp/mt_noalign.out 2>&1 

^C


In [1]:
!cat train_overnight.py


"""
RFT-LM Overnight Training — C4 Pretraining on 3×L40S

Trains two models on real C4 text:
  1. RFT-LM (Transformer + RFT memory layer + OCR) 
  2. Baseline Transformer (same size, no memory)

Both use the same data, same hyperparams, same training budget.
The comparison shows whether the memory layer helps.

Usage:
    # Run on GPUs 3,4,5 overnight (~8-10 hours)
    CUDA_VISIBLE_DEVICES=3,4,5 python train_overnight.py \
        --data_path ./data/c4_train.jsonl \
        --outdir ./runs_lm/overnight \
        --model rft_lm \
        --n_gpus 3

    # Then run baseline on GPUs 3,4,5
    CUDA_VISIBLE_DEVICES=3,4,5 python train_overnight.py \
        --data_path ./data/c4_train.jsonl \
        --outdir ./runs_lm/overnight \
        --model baseline \
        --n_gpus 3
"""

import argparse
import json
import math
import os
import random
import time
from typing import List, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
fro

In [2]:
!cat eval_ruler_niah.py


"""
eval_ruler_niah.py — Paper-grade RULER-style NIAH evaluator for local checkpoints.

Adds:
  - Depth-stratified sweep (single-depth or grid)
  - Multi-key / multi-needle NIAH (MK-NIAH)
  - Wilson confidence intervals
  - RFT ablation hooks (disable memory, neutralize OCR influence)
"""

import argparse
import json
import math
import os
import random
import uuid
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import torch

from RFT_LM import BaselineTransformerLM, MemoryBank, RFTLM, MTLM


def string_match_all_binary(pred: str, refs: List[str]) -> bool:
    if not refs:
        return False
    p = pred.lower()
    return all(r.lower() in p for r in refs)


def wilson_ci(k: int, n: int, z: float = 1.96) -> Tuple[float, float]:
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    denom = 1.0 + (z * z) / n
    center = (p + (z * z) / (2 * n)) / denom
    margin = (z / denom) * math.sqrt((p * (1 - p) / n) + ((z * z) / (4 * n * n)))
    lo = max(0.0, center

In [3]:
!cat niah_sanity.py


"""
niah_sanity.py — Minimal sanity test for NIAH eval pipeline.

Goal: isolate whether 0%/0% accuracy is (a) a pipeline bug or (b) the
125M C4-only models genuinely cannot follow the NIAH prompt template.

Test design (trivially easy):
  - Very short context (e.g. 256 tokens of distractor)
  - Needle placed immediately before the query (depth=1.0)
  - Numeric value, single-key single-value
  - Greedy decode 20 new tokens

If BOTH models get 0 hits here, the problem is prompt/template (models
cannot follow this instruction format from C4 pretraining alone).
If baseline gets a reasonable hit rate, the pipeline works and the full
eval is meaningful.

We also print the raw generations for each trial so you can eyeball what
the model actually produces.
"""

import argparse
import random
from pathlib import Path

import torch
from transformers import AutoTokenizer

from RFT_LM import MemoryBank
from eval_ruler_niah import (
    build_mk_niah_sample,
    generate_greedy,
    load_baseline_mo

In [4]:
!cat niah_batch.py


import json
"""
niah_batch.py â€” RULER-faithful NIAH training batches for RFT-LM.

Uses the EXACT RULER S-NIAH template (Hsieh et al., 2024) so that
training and evaluation use the same format. This makes results
directly comparable to Titans and other papers citing RULER.

RULER template:
  Instruction: "Some special magic {type} are hidden within the
    following text. Make sure to memorize it. I will quiz you
    about the {type} afterwards."
  Needle: "The special magic {type} for {key} is: {value}."
  Query: "What are all the special magic {type} for {key}
    mentioned in the provided text?"
  Answer prefix: "The special magic {type} for {key} mentioned
    in the provided text are"

Layout per example (2-chunk for memory training):
  Chunk 0 (context):
    [RULER instruction prefix]
    [C4 distractor text with RULER-format needles embedded]
    -> Processed through model, written to memory bank.

  Chunk 1 (query):
    [C4 distractor text]
    [RULER query + answer prefix]
  

In [6]:
# 1. Is v11 still running?
!ps aux | grep python | grep -v grep

# 2. Training progress
!tail -5 /zeng_gk/Amine/Huawei\ Challenge/RFT/AmineHL/*/rft_lm/latest_metrics.json 2>/dev/null || tail -5 /zeng_gk/Amine/Huawei\ Challenge/RFT/AmineHL/*/rft_lm/v11*.log 2>/dev/null

# 3. What checkpoints exist for v10 and v11?
!ls -lh /zeng_gk/Amine/Huawei\ Challenge/RFT/AmineHL/*/rft_lm/*.pt 2>/dev/null

# 4. Are PG essays downloaded?
!ls -lh /zeng_gk/Amine/Huawei\ Challenge/RFT/AmineHL/paulgraham_essays* 2>/dev/null || ls -lh /zeng_gk/Amine/Huawei\ Challenge/RFT/AmineHL/data/paul* 2>/dev/null


root         31  4.8  0.0 759952 140652 ?       Sl   Mar22 1732:14 /opt/conda/bin/python3.8 /opt/conda/bin/jupyter-lab --allow-root --notebook-dir=/zeng_gk --LabApp.base_url=jupyter/https%3A%2F%2F10.251.171.18%3A47884
root      49181  0.0  0.2 11964284 679744 ?     Ssl  Apr15   0:38 /opt/conda/envs/qwen/bin/python -m ipykernel_launcher -f /root/.local/share/jupyter/runtime/kernel-72bafc47-e3e0-4770-89da-9f61834c92fb.json
root      49203  0.0  0.0 818960 60348 ?        Ssl  Apr15   0:04 /opt/conda/bin/python3.8 -m ipykernel_launcher -f /root/.local/share/jupyter/runtime/kernel-d9905770-324f-46e3-9fbe-bf4110fbf71a.json
root      49703  100  1.4 35455952 3835844 ?    Sl   Apr15 939:16 python train_overnight.py --data_path data/c4_train.jsonl --outdir pretrain_350m_ruler_seed42 --model rft_lm --tokenizer_path gpt2_tokenizer --vocab_size 50257 --d_model 1024 --n_layers 24 --n_heads 16 --ff_mult 4 --window_size 512 --memory_layer_idx 12 --mem_top_m 96 --ocr_dim 384 --total_seq_len 4096 --chu

In [1]:
!tail -10 logs/350m_v12.log  # or whatever it's called
!ls -lat pretrain_350m_v12_ruler_full_seed42/rft_lm/ | head -5

	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
  step 25100 | SYNTH loss 11.7501 | router_ce 8.177 | r@1 0.750 | r@M 1.000 | ptr 0.750 | fused 0.750 | ratio 0.04
  step 25200 | loss 4.3906 | avg 32.6450 | lr 9.41e-05 | 3237 tok/s | peak 11933MB mem=4096 | eta 1993.3m
  step 25300 | loss 4.3948 | avg 29.1452 | lr 9.41e-05 | 3239 tok/s | peak 11933MB mem=4096 | eta 2015.8m
  step 25400 | NIAH loss 21.1131 | lm_acc 0.250 | emb_acc 1.000 | emb_loss 0.060 | r@M 1.000 | fused 1.000
  step 25500 | NIAH loss 447.7385 | lm_acc 0.000 | emb_acc 0.000 | emb_loss 19.454 | r@M 1.000 | fused 0.000
total 14103267
-rw-r--r--. 1 root root      85696 Ap